# 3.4 Code Brief: Evaluating Tree-Based Models

This notebook contains a condensed reference of the key code patterns from notebook 3.4. Use it as a quick reference.

## Key Pattern: Instantiate → Fit → Predict

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Same three lines for any model:
model = ModelClass(**params)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
```

## Setup and Data Preparation

In [ ]:
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
course3_models = '/models/'

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score, ConfusionMatrixDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ARTIFACT_DIR = f'{project_path}{course3_models}'
feature_columns = joblib.load(f'{ARTIFACT_DIR}feature_columns.pkl')
train_medians   = joblib.load(f'{ARTIFACT_DIR}train_medians.pkl')

# Load TEST data only
test_df = pd.read_csv(f'{project_path}{data_filepath}testing.csv')
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

test_enc = pd.get_dummies(test_df[numeric_features + categorical_features],
                          columns=categorical_features, drop_first=True)
# Reconstruct the exact training feature set: add missing dummies as 0, drop unseen ones
test_enc = test_enc.reindex(columns=feature_columns, fill_value=0)
# Impute with TRAIN medians (column-aligned), never test's own
test_enc = test_enc.fillna(train_medians)

X_test, y_test = test_enc, test_df['DEPARTED']
print(f"Test data prepared: {X_test.shape[0]:,} samples | {X_test.shape[1]} features")

## Load Pre-Trained Models

In [ ]:
models = {}
predictions = {}
probabilities = {}
model_filenames = {
    'Decision Tree': 'dt_tuned_f1.pkl',
    'Random Forest': 'rf_tuned_f1.pkl',
    'XGBoost': 'xgb_tuned_f1.pkl'
}
for name, filename in model_filenames.items():
    model_path = f'{project_path}{course3_models}{filename}'
    model = joblib.load(model_path)
    models[name] = model
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)[:, 1]
print(f"Models loaded: {list(models.keys())}")

## Precision-Recall Curves

In [ ]:
plt.figure(figsize=(10, 7))
baseline_prevalence = y_test.sum() / len(y_test)
for name, prob in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap_score = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f'{name} (AP = {ap_score:.2f})')
plt.plot([0, 1], [baseline_prevalence, baseline_prevalence], linestyle='--',
         color='gray', label='Baseline (Class Prevalence)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Tree-Based Models')
plt.legend()
plt.grid(True)
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.show()

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices for Tree-Based Models', fontsize=16)
for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Key Takeaways

- Standard accuracy can be misleading on imbalanced data (this dataset: ~15% departure rate)
- **PR curves** focus on the trade-off between Precision and Recall — more informative than ROC-AUC here
- **Confusion matrices** show the raw TP/FP/FN/TN counts, useful for spotting where a model fails on the minority class

**Next:** Module 4 — Model Comparison